# Distributed LLM Fine-Tuning with Ray Train

This notebook demonstrates how to fine-tune large language models (LLMs) using Ray Train with DeepSpeed integration. You'll learn how to:

- Set up a distributed training environment (local Ray or Anyscale)
- Configure DeepSpeed ZeRO for memory-efficient training
- Use LoRA/QLoRA for parameter-efficient fine-tuning
- Scale training across multiple GPUs (2-8 GPUs)
- Manage checkpoints and resume training

## Prerequisites

- 2-8 GPUs with at least 16GB VRAM each (for 7B models)
- Python 3.9+
- HuggingFace account for gated models (optional)

## 1. Installation

Install the required packages:

In [ ]:
!pip install -q "ray[train]" torch transformers datasets accelerate deepspeed peft bitsandbytes

## 2. Setup and Configuration

In [ ]:
import os
import sys

# Add utils to path
sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    get_recommended_workers,
    get_deepspeed_config,
    init_ray,
    ClusterMode,
    get_scaling_config,
    get_run_config,
    create_runtime_env,
)

In [ ]:
# Check GPU availability
print_gpu_status()

# Get recommended number of workers
NUM_WORKERS = get_recommended_workers()
print(f"\nRecommended workers: {NUM_WORKERS}")

### Choose Your Deployment Mode

**Option A: Local Ray Cluster** - Use this if running on a local machine or single node with multiple GPUs.

**Option B: Anyscale Platform** - Use this for managed Ray clusters with autoscaling and cloud storage.

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL  # Change to ClusterMode.ANYSCALE for Anyscale

# Model and training config
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small model for demo
# MODEL_NAME = "microsoft/phi-2"  # Alternative: 2.7B model
# MODEL_NAME = "meta-llama/Llama-2-7b-hf"  # Larger model (needs more GPUs)

DATASET_NAME = "tatsu-lab/alpaca"  # Instruction-following dataset
MAX_LENGTH = 512
BATCH_SIZE_PER_DEVICE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1

# LoRA config
USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Storage paths
if CLUSTER_MODE == ClusterMode.LOCAL:
    STORAGE_PATH = "./runs/finetuning"
else:
    # For Anyscale, use cloud storage
    STORAGE_PATH = "s3://your-bucket/ray-checkpoints/finetuning"

## 3. Initialize Ray Cluster

In [ ]:
# Create runtime environment with required packages
runtime_env = create_runtime_env(
    pip_packages=["transformers", "datasets", "accelerate", "peft", "bitsandbytes"],
)

# Initialize Ray
init_ray(
    mode=CLUSTER_MODE,
    runtime_env=runtime_env,
)

## 4. Prepare Dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset
dataset = load_dataset(DATASET_NAME, split="train")

print(f"Dataset size: {len(dataset)} samples")
print(f"Sample: {dataset[0]}")

In [ ]:
def format_alpaca(example):
    """Format Alpaca dataset into instruction format."""
    if example.get("input", ""):
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    return {"text": text}


def tokenize_function(examples):
    """Tokenize examples."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )


# Format and tokenize dataset
dataset = dataset.map(format_alpaca)
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

# For demo, use subset
tokenized_dataset = tokenized_dataset.shuffle(seed=42).select(range(min(5000, len(tokenized_dataset))))

print(f"Tokenized dataset size: {len(tokenized_dataset)} samples")

## 5. Define Training Function

The training function runs on each worker. Ray Train automatically handles:
- Distributed data loading
- Model parallelism via DeepSpeed
- Checkpointing and fault tolerance

In [ ]:
import ray
from ray import train
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig, CheckpointConfig


def train_func(config):
    """Training function that runs on each worker."""
    import torch
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        DataCollatorForLanguageModeling,
    )
    from peft import LoraConfig, get_peft_model, TaskType
    from datasets import Dataset

    # Get config values
    model_name = config["model_name"]
    use_lora = config["use_lora"]
    learning_rate = config["learning_rate"]
    num_epochs = config["num_epochs"]
    batch_size = config["batch_size_per_device"]
    gradient_accumulation = config["gradient_accumulation_steps"]

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # Apply LoRA if enabled
    if use_lora:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    # Get dataset from Ray object store
    train_dataset = train.get_dataset_shard("train")
    train_dataset_hf = train_dataset.to_pandas()
    train_dataset_hf = Dataset.from_pandas(train_dataset_hf)

    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    # Training arguments
    training_args = TrainingArguments(
        output_dir="./output",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=learning_rate,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        deepspeed=config.get("deepspeed_config"),
        gradient_checkpointing=True,
        ddp_find_unused_parameters=False,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_hf,
        data_collator=data_collator,
    )

    # Train
    trainer.train()

    # Report metrics to Ray Train
    train.report(
        {"loss": trainer.state.log_history[-1].get("loss", 0)},
    )

## 6. Configure and Launch Training

In [ ]:
# DeepSpeed configuration
# Stage 2: Partition optimizer states and gradients across GPUs
# Stage 3: Also partition model parameters (for very large models)
deepspeed_config = get_deepspeed_config(
    stage=2,  # Use stage 3 for models >7B parameters
    offload_optimizer=False,  # Set True to offload to CPU (saves GPU memory)
)

# Training config passed to workers
train_config = {
    "model_name": MODEL_NAME,
    "use_lora": USE_LORA,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "batch_size_per_device": BATCH_SIZE_PER_DEVICE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "deepspeed_config": deepspeed_config,
}

print("Training configuration:")
for k, v in train_config.items():
    if k != "deepspeed_config":
        print(f"  {k}: {v}")

In [ ]:
# Convert HuggingFace dataset to Ray Dataset for distributed loading
ray_dataset = ray.data.from_huggingface(tokenized_dataset)

print(f"Ray Dataset: {ray_dataset.count()} samples")

In [ ]:
# Create scaling config
scaling_config = get_scaling_config(
    num_workers=NUM_WORKERS,
    use_gpu=True,
)

# Create run config with checkpointing
run_config = get_run_config(
    name="llm-finetuning",
    storage_path=STORAGE_PATH,
    checkpoint_config={"num_to_keep": 2},
    failure_config={"max_failures": 3},
)

print(f"Scaling config: {NUM_WORKERS} workers with GPU")
print(f"Storage path: {STORAGE_PATH}")

In [ ]:
# Create TorchTrainer
trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    train_loop_config=train_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": ray_dataset},
)

print("TorchTrainer created successfully")

In [ ]:
# Launch distributed training
print(f"Starting distributed training on {NUM_WORKERS} GPUs...")
print(f"Model: {MODEL_NAME}")
print(f"LoRA: {'Enabled' if USE_LORA else 'Disabled'}")
print("="*50)

result = trainer.fit()

print("="*50)
print("Training completed!")
print(f"Final loss: {result.metrics.get('loss', 'N/A')}")
print(f"Checkpoint path: {result.checkpoint}")

## 7. Load and Test the Fine-tuned Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

# Load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# If using LoRA, load the adapter
if USE_LORA and result.checkpoint:
    # Note: In production, you'd load from the checkpoint path
    print("Note: Load LoRA weights from checkpoint for inference")

# Create pipeline for inference
pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
)

In [ ]:
# Test the model
test_prompt = """### Instruction:
Write a Python function that calculates the factorial of a number.

### Response:
"""

output = pipe(test_prompt)
print(output[0]["generated_text"])

## 8. Cleanup

In [ ]:
from utils import shutdown_ray

shutdown_ray()
print("Ray cluster shutdown complete")

## Next Steps

- **Scale up**: Increase `NUM_WORKERS` to use more GPUs
- **Larger models**: Try `meta-llama/Llama-2-7b-hf` with DeepSpeed Stage 3
- **QLoRA**: Add quantization for even more memory efficiency
- **Deploy**: See `llm_serving.ipynb` for deploying your fine-tuned model

## Resources

- [Ray Train Documentation](https://docs.ray.io/en/latest/train/train.html)
- [DeepSpeed ZeRO Tutorial](https://docs.ray.io/en/latest/train/examples/pytorch/deepspeed_finetune/README.html)
- [PEFT/LoRA Documentation](https://huggingface.co/docs/peft)
- [Anyscale Platform](https://docs.anyscale.com/)